<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/correction_timeseries_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation des bibliothèques

Installez (ou mettez à jour) les bibliothèques nécessaires au notebook : **keras** et **keras-hub**.
Utilisez une commande `pip` dans une cellule notebook, en mode silencieux si possible.

In [ ]:
# préparer l’environnement d’exécution (Colab / Jupyter) avec les packages requis.
!pip install keras keras-hub --upgrade -q

### Configuration du backend Keras

Configurez Keras pour utiliser le backend **TENSORFLOW** en définissant la variable d’environnement appropriée **avant** d’importer Keras.

In [ ]:
# Keras peut s’appuyer sur plusieurs backends (TensorFlow/JAX/Torch selon version).
import os
# Ici on force tensorflow.
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Timeseries forecasting

### Different kinds of timeseries tasks

### A temperature forecasting example

### Récupération du jeu de données

Téléchargez l’archive du dataset **Jena Climate 2009–2016** (format `.zip`) depuis l’URL fournie dans le cours, puis décompressez-la afin d’obtenir le fichier CSV.

In [ ]:
# CSV brut qui servira à l’exemple de prévision de température.
!wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
!unzip jena_climate_2009_2016.csv.zip

### Lecture et inspection du CSV

Chargez le fichier CSV en mémoire, puis :
- séparez les lignes,
- récupérez l’en-tête (noms de colonnes),
- affichez l’en-tête et le nombre total d’exemples (lignes de données, sans l’en-tête).

In [ ]:
import os

fname = os.path.join("jena_climate_2009_2016.csv")

with open(fname) as f:
    data = f.read()

lines = data.split("\n")
header = lines[0].split(",")
lines = lines[1:]
# première exploration du format des données.
print(header)
print(len(lines))

### Mise en forme NumPy

Construisez :
- un tableau `temperature` contenant uniquement la température cible,
- un tableau `raw_data` contenant toutes les variables numériques (features),
en convertissant les valeurs texte du CSV en `float`.

Indication : parcourez les lignes et alimentez progressivement les tableaux.

In [ ]:
import numpy as np
# prépare la cible (température) et les entrées (variables météo) au format exploitable par les modèles
temperature = np.zeros((len(lines),))
raw_data = np.zeros((len(lines), len(header) - 1))

for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]
    temperature[i] = values[1]
    raw_data[i, :] = values[:]

### Visualisation globale

Tracez l’évolution de la température sur l’ensemble de la série temporelle avec Matplotlib.

In [ ]:
from matplotlib import pyplot as plt
# visualiser la tendance long-terme / saisonnalité
plt.plot(range(len(temperature)), temperature)

### Zoom sur une courte fenêtre

Tracez la température sur une petite fenêtre temporelle (par exemple les **1440 premiers points**) afin d’observer la dynamique à court terme.

In [ ]:
# mieux voir la variation journalière/hebdo.
plt.plot(range(1440), temperature[:1440])

### Découpage des données

Calculez le nombre d’exemples pour :
- l’entraînement : 50%,
- la validation : 25%,
- le test : le reste,

puis affichez les trois valeurs.

In [ ]:
# sépare chronologiquement pour éviter les fuites temporelles
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples
print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

#### Préparation des données : normalisation

Calculez la moyenne et l’écart-type des features sur la partie **train uniquement**.
Puis normalisez `raw_data`.

In [ ]:
# standardisation sans fuite (on n’utilise pas val/test pour calculer μ, σ)
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean
std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

### Principe “fenêtre → cible future” (séries temporelles)

L’idée est de transformer une **série temporelle** en un problème d’apprentissage supervisé.

- **Fenêtre (window)** : un ensemble de valeurs passées consécutives → **entrée du modèle**
- **Cible future (target)** : une valeur située plus loin dans le temps → **sortie à prédire**

---

#### Exemple simple

Série :  
`0, 1, 2, 3, 4, 5, 6, 7, 8, 9`

Avec :
- longueur de fenêtre = 3  
- horizon de prédiction = 1 pas

On obtient :

- `[0, 1, 2]` → `3`
- `[1, 2, 3]` → `4`
- `[2, 3, 4]` → `5`
- etc.

Le modèle apprend donc à prédire une **valeur future** à partir des **valeurs passées**.

---

#### Application au cas météo (exemple Jena)

- **Fenêtre** : plusieurs mesures météo passées (température, pression, humidité…)
- **Cible future** : température à un horizon donné (ex : 24h plus tard)

Ce mécanisme permet d’utiliser les réseaux de neurones (Dense, CNN, RNN, LSTM, etc.) pour faire de la **prévision temporelle**.

### Démonstration : création d’un dataset temporel

Créez une séquence d’entiers (0 à 9), puis utilisez `keras.utils.timeseries_dataset_from_array` pour générer :
- des fenêtres (séquences) de longueur 3 comme entrées,
- la valeur 3 pas plus loin comme cible.

Affichez ensuite, pour chaque batch, les séquences d’entrée et leur cible associée.

In [ ]:
import numpy as np
import keras

int_sequence = np.arange(10)
dummy_dataset = keras.utils.timeseries_dataset_from_array(
    data=int_sequence[:-3],
    targets=int_sequence[3:],
    sequence_length=3,
    batch_size=2,
)

for inputs, targets in dummy_dataset:
    for i in range(inputs.shape[0]):
        print([int(x) for x in inputs[i]], int(targets[i]))

### Création des datasets temporels (train/val/test)

Définissez les paramètres :
- `sampling_rate`
- `sequence_length`
- `delay`
- `batch_size`

Puis construisez `train_dataset`, `val_dataset` et `test_dataset` avec `keras.utils.timeseries_dataset_from_array` en respectant :
- les entrées : `raw_data` (jusqu’à `-delay`),
- les cibles : `temperature` décalée de `delay`,
- des indices `start_index` / `end_index` cohérents avec le découpage train/val/test,
- un mélange (`shuffle=True`) et un `batch_size` fixé.

In [ ]:
sampling_rate = 6
sequence_length = 120
# définissent la fenêtre (ex : 120 points espacés)
delay = sampling_rate * (sequence_length + 24 - 1)
# définit à quel horizon on prédit la température
batch_size = 256
# on prépare la cible (température) et les entrées (variables météo) au format exploitable par les modèles
train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples,
)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples,
)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples,
)

### Vérification des dimensions

Itérez sur `train_dataset` et affichez la forme (`shape`) :
- des entrées `samples`,
- des cibles `targets`,
puis arrêtez après le premier batch.

In [ ]:
for samples, targets in train_dataset:
    print("samples shape:", samples.shape)
    print("targets shape:", targets.shape)
    break # sanity check : (batch, timesteps, features) et (batch,) (ou proche)

#### Baseline naïve (sans apprentissage)

Implémentez une fonction `evaluate_naive_method(dataset)` qui :
- prédit la température future en utilisant simplement la dernière valeur de température disponible dans la séquence d’entrée,
- remet la prédiction à l’échelle originale (annule la normalisation),
- calcule l’erreur absolue moyenne (MAE) sur tout le dataset.

Affichez ensuite la MAE sur validation puis sur test.

In [ ]:
def evaluate_naive_method(dataset):
    total_abs_err = 0.0
    samples_seen = 0
    for samples, targets in dataset:
        preds = samples[:, -1, 1] * std[1] + mean[1]
        total_abs_err += np.sum(np.abs(preds - targets))
        samples_seen += samples.shape[0]
    return total_abs_err / samples_seen

print(f"Validation MAE: {evaluate_naive_method(val_dataset):.2f}")
print(f"Test MAE: {evaluate_naive_method(test_dataset):.2f}")

Dans le code, la prédiction correspond à :

- prendre la **dernière valeur de température disponible dans la séquence d’entrée** ;
- annuler la normalisation pour revenir à l’échelle réelle.

Autrement dit :

> On prédit que la température future sera égale à la température actuelle.

---

**Pourquoi cette stratégie n’est-elle pas absurde ?**

La température :
- évolue de manière progressive,
- présente une forte corrélation temporelle,
- ne change généralement pas brutalement.

Cette stratégie dite de *persistance* peut donc donner des résultats déjà raisonnables.

---

**Pourquoi cette baseline est-elle essentielle ?**

Elle sert de **référence minimale**.

Un modèle plus complexe (Dense, Conv1D, LSTM, etc.) doit impérativement faire **mieux que cette baseline**.  
Sinon, cela signifie que le modèle :

- n’apporte aucune valeur ajoutée,
- est mal configuré,
- ou apprend mal le problème.

#### Premier modèle de ML : réseau dense

Construisez un modèle Keras qui :
- prend en entrée une séquence de forme `(sequence_length, nb_features)`,
- aplatie la séquence (`Flatten`),
- applique une couche dense cachée (ex : 16 neurones, ReLU),
- produit une sortie scalaire (température prédite).

Compilez avec `adam`, `mse` et la métrique `mae`.
Entraînez 10 époques avec validation, et sauvegardez le meilleur modèle via `ModelCheckpoint`.
Enfin, rechargez le meilleur modèle et évaluez-le sur le test.

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Flatten()(inputs)
x = layers.Dense(16, activation="relu")(x)
outputs = layers.Dense(1)(x)
# modèle de base “non temporel” (il casse l’ordre interne en aplatissant)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_dense.keras", save_best_only=True)
]
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks,
)

model = keras.models.load_model("jena_dense.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

### Courbes d’apprentissage

À partir de l’objet `history`, tracez :
- la MAE d’entraînement en fonction des époques,
- la MAE de validation en fonction des époques,
avec un titre, une légende, et l’affichage du graphique.

In [ ]:
import matplotlib.pyplot as plt

loss = history.history["mae"]
val_loss = history.history["val_mae"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "r--", label="Training MAE")
plt.plot(epochs, val_loss, "b", label="Validation MAE")
# diagnostic sur surapprentissage / sous-apprentissage
plt.title("Training and validation MAE")
plt.legend()
plt.show()

#### Modèle convolutionnel 1D

Construisez un modèle Keras 1D Conv pour séries temporelles :
- plusieurs `Conv1D` + `MaxPooling1D`,
- puis `GlobalAveragePooling1D`,
- puis une `Dense(1)` en sortie.

Compilez (adam/mse/mae), entraînez 10 époques avec `ModelCheckpoint`,
rechargez le meilleur modèle et évaluez sur test.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
# la convolution extrait des motifs locaux dans le temps
x = layers.Conv1D(8, 24, activation="relu")(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 12, activation="relu")(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 6, activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_conv.keras", save_best_only=True)
]
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks,
)

model = keras.models.load_model("jena_conv.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

### Modèle récurrent : LSTM

Construisez un modèle séquentiel fonctionnel avec :
- une couche `LSTM(16)` qui lit la séquence,
- une couche `Dense(1)` en sortie.

Ajoutez `ModelCheckpoint`, entraînez 10 époques et évaluez sur le test après rechargement du meilleur modèle.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm.keras", save_best_only=True)
]
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks,
)

model = keras.models.load_model("jena_lstm.keras")
print("Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

#### Compréhension d’un RNN : calcul manuel

Simulez la propagation d’un RNN simple “à la main” avec NumPy :
- générez une séquence d’entrées aléatoires `(timesteps, input_features)`,
- initialisez un état caché `state_t`,
- définissez des matrices de poids `W`, `U` et un biais `b`,
- pour chaque pas de temps, calculez :
  output_t = np.tanh(np.dot(W, input_t) + np.dot(U, state_t) + b)
- stockez les sorties successives et construisez une séquence finale.

In [ ]:
import numpy as np

timesteps = 100
input_features = 32
output_features = 64
inputs = np.random.random((timesteps, input_features))
state_t = np.zeros((output_features,))
W = np.random.random((output_features, input_features))
U = np.random.random((output_features, output_features))
b = np.random.random((output_features,))
successive_outputs = []
for input_t in inputs:
    output_t = np.tanh(np.dot(W, input_t) + np.dot(U, state_t) + b)
    successive_outputs.append(output_t)
    state_t = output_t
final_output_sequence = np.concatenate(successive_outputs, axis=0)

#### SimpleRNN dans Keras (longueur variable)

Définissez un tenseur d’entrée Keras de forme `(None, num_features)` (longueur temporelle variable),
puis appliquez une couche `SimpleRNN(16)` pour obtenir la représentation finale.

In [ ]:
num_features = 14
inputs = keras.Input(shape=(None, num_features))
# None signifie “nombre de pas de temps variable”.
outputs = layers.SimpleRNN(16)(inputs)

### SimpleRNN : sortie finale uniquement

Créez une entrée de forme `(steps, num_features)`,
appliquez `SimpleRNN(16, return_sequences=False)` et affichez la forme du tenseur de sortie.

In [ ]:
num_features = 14
steps = 120
inputs = keras.Input(shape=(steps, num_features))
outputs = layers.SimpleRNN(16, return_sequences=False)(inputs)
# une seule représentation (dernier état)
print(outputs.shape)

### SimpleRNN : sortie séquentielle

Reprenez le même exemple, mais avec `return_sequences=True`,
puis affichez la forme du tenseur de sortie.

In [ ]:
num_features = 14
steps = 120
inputs = keras.Input(shape=(steps, num_features))
outputs = layers.SimpleRNN(16, return_sequences=True)(inputs)
# on garde la séquence complète des états cachés
print(outputs.shape)

### Empilement de couches récurrentes

Construisez un petit réseau avec plusieurs `SimpleRNN` empilées :
- les couches intermédiaires doivent avoir `return_sequences=True`,
- la dernière couche renvoie uniquement la sortie finale.

In [ ]:
inputs = keras.Input(shape=(steps, num_features))
# pour empiler, il faut transmettre une séquence aux couches suivantes
x = layers.SimpleRNN(16, return_sequences=True)(inputs)
x = layers.SimpleRNN(16, return_sequences=True)(x)
outputs = layers.SimpleRNN(16)(x)

#### Lutter contre l’overfitting : dropout récurrent

Construisez un modèle avec :
- `LSTM(32, recurrent_dropout=0.25)`,
- un `Dropout(0.5)` après la LSTM,
- une sortie `Dense(1)`.

Entraînez plus longtemps (ex : 50 époques) avec `ModelCheckpoint` pour sauvegarder le meilleur modèle.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(32, recurrent_dropout=0.25)(inputs)
# recurrent_dropout agit sur les connexions récurrentes (mémoire), utile contre overfitting
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "jena_lstm_dropout.keras", save_best_only=True
    )
]
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=callbacks,
)

#### Empilement de GRU (avec dropout)

Construisez un modèle de type :
- `GRU(32, recurrent_dropout=0.5, return_sequences=True)`,
- puis `GRU(32, recurrent_dropout=0.5)` (sans séquences),
- puis `Dropout(0.5)` et `Dense(1)`.

Entraînez 50 époques avec `ModelCheckpoint`, rechargez le meilleur modèle et évaluez sur test.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
# GRU est une variante plus simple que LSTM, souvent efficace.
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "jena_stacked_gru_dropout.keras", save_best_only=True
    )
]
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=callbacks,
)
model = keras.models.load_model("jena_stacked_gru_dropout.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

#### RNN bidirectionnel

Construisez un modèle avec :
- une couche `Bidirectional(LSTM(16))` appliquée à la séquence,
- une sortie `Dense(1)`.

Compilez et entraînez (ex : 10 époques) avec validation.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Bidirectional(layers.LSTM(16))(inputs)
# la couche lit la séquence dans les deux sens (utile quand toute la séquence est disponible)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
)